# ML Zoomcamp 2026 — Homework 3: Classification

Dataset: `course_lead_scoring_2026.csv`

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [4]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation
Categorical missing → `'NA'`, numerical missing → `0.0`.

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
categorical, numerical

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'])

## Q1. Mode of `industry`

In [7]:
df.industry.mode()[0], df.industry.value_counts().head()

('technology',
 industry
 technology    1173
 retail         925
 healthcare     840
 finance        720
 education      639
 Name: count, dtype: int64)

## Q2. Biggest correlation

In [8]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [('interaction_count', 'lead_score'), ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'), ('annual_income', 'interaction_count')]
{f'{a} & {b}': round(corr.loc[a, b], 4) for a, b in pairs}

{'interaction_count & lead_score': np.float64(0.9157),
 'number_of_courses_viewed & lead_score': np.float64(0.7572),
 'number_of_courses_viewed & interaction_count': np.float64(0.7216),
 'annual_income & interaction_count': np.float64(0.1228)}

## Split

In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
df_train, df_val, df_test = [d.reset_index(drop=True) for d in (df_train, df_val, df_test)]
y_train, y_val, y_test = df_train.converted.values, df_val.converted.values, df_test.converted.values
for d in (df_train, df_val, df_test):
    del d['converted']
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q3. Mutual information (training set)

In [11]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Q4. Logistic regression accuracy

In [12]:
def train_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    # float(): np.float64 rounds half-to-even (0.645 -> 0.64); Python float gives 0.65 like the official checker
    return float((model.predict(X_val) == y_val).mean())

features = categorical + numerical
base_acc = train_eval(features)
base_acc, round(base_acc, 2)

(0.645, 0.65)

## Q5. Feature elimination
Difference = original accuracy − accuracy without the feature.

In [13]:
diffs = {f: base_acc - train_eval([x for x in features if x != f]) for f in features}
pd.Series(diffs).sort_values()

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [14]:
{f: diffs[f] for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']}

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

## Q6. Regularization

In [15]:
{C: round(train_eval(features, C), 3) for C in [0.000001, 0.00001, 0.0001, 0.001]}

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}